# Within-Corpus Spam Classification — Organized Notebook

This notebook runs **one corpus and one feature configuration at a time**. Select the corpus (`"A"` or `"B"`) and the configuration in the settings cell below, then restart the kernel and run all cells whenever you change either of them.

The evaluation setup is kept consistent with the reported experiments: five classifiers, the existing hyperparameters, `RepeatedStratifiedKFold` with 10 folds × 3 repetitions, and accuracy / precision / recall / F1 reporting.

**Input expectation:** the two experiment-ready 30,000-row corpus files `dataset_a_within_30000_ids.pkl` and `dataset_b_within_30000_ids.pkl` must exist in the directory set by `DATA_DIR` (default: the current working directory). Each file contains the preprocessed `text` column, the six metadata columns, the binary `label` column (1 = spam, 0 = legitimate) and the record identifiers. This notebook does not rebuild those features from raw email messages.

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

## 2. Settings — choose one configuration

In [ ]:
# Choose the corpus: "A" or "B"
DATASET = "A"

# Choose exactly one: "Text", "Metadata", or "Text + Metadata"
CONFIGURATION = "Text"

# Directory that contains the corpus files (default: the current working directory).
DATA_DIR = "."

DATA_FILES = {
    "A": "dataset_a_within_30000_ids.pkl",
    "B": "dataset_b_within_30000_ids.pkl",
}
EXPECTED_ROWS = 30000

# Shared experiment settings
RANDOM_STATE = 42
N_SPLITS = 10
N_REPEATS = 3
TFIDF_MAX_FEATURES = 20000
TFIDF_NGRAM_RANGE = (1, 2)

if DATASET not in DATA_FILES:
    raise ValueError(f"DATASET must be one of {sorted(DATA_FILES)}; got {DATASET!r}.")

VALID_CONFIGURATIONS = {"Text", "Metadata", "Text + Metadata"}
if CONFIGURATION not in VALID_CONFIGURATIONS:
    raise ValueError(
        f"CONFIGURATION must be one of {sorted(VALID_CONFIGURATIONS)}; got {CONFIGURATION!r}."
    )

DATA_PATH = os.path.join(DATA_DIR, DATA_FILES[DATASET])

metadata_features = [
    "num_exclamations",
    "num_dollars",
    "num_urls",
    "num_emails",
    "num_hashes",
    "num_digits",
]

print("Selected corpus:", DATASET)
print("Selected configuration:", CONFIGURATION)
print("Input path:", DATA_PATH)

## 3. Load and validate the experiment-ready dataset

The corpus file for the selected dataset is read from `DATA_DIR`. This cell checks the expected number of rows, the required columns, the labels and the record identifiers. It deliberately does not impute or modify missing values; it reports them and stops if any are found, so the preprocessing decision is not guessed.

In [ ]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(
        f"Could not find the corpus file at {DATA_PATH!r}. "
        f"Place {DATA_FILES[DATASET]!r} in the directory given by DATA_DIR "
        f"(currently {os.path.abspath(DATA_DIR)!r}) or change DATA_DIR in the settings cell."
    )

df = pd.read_pickle(DATA_PATH).copy()

# Accommodate the alternative text-column name used in an earlier notebook cell.
if "text" not in df.columns and "text_combined" in df.columns:
    df = df.rename(columns={"text_combined": "text"})

if len(df) != EXPECTED_ROWS:
    raise ValueError(f"Expected {EXPECTED_ROWS} rows in {DATA_FILES[DATASET]!r}, found {len(df)}.")

required_columns = ["text", "label", *metadata_features]
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(
        "The loaded dataframe is missing required columns: "
        f"{missing_columns}. Available columns: {df.columns.tolist()}"
    )

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

# Record identifiers (the file name carries '_ids'): report them and check that they are unique.
ID_CANDIDATES = ["id", "record_id", "sample_id", "original_id", "row_id"]
id_columns = [column for column in ID_CANDIDATES if column in df.columns]
if id_columns:
    for column in id_columns:
        print(f"Identifier column {column!r}: {df[column].nunique()} unique values for {len(df)} rows")
        if df[column].nunique() != len(df):
            print(f"WARNING: identifier column {column!r} is not unique.")
else:
    print("NOTE: no identifier column named any of", ID_CANDIDATES, "was found; check the 'Columns' line above.")

print("\nMissing values in required columns:")
print(df[required_columns].isna().sum())
print("\nLabel counts:")
print(df["label"].value_counts(dropna=False))

if df[required_columns].isna().any().any():
    raise ValueError(
        "Missing values were found in required columns. Resolve them using the actual documented "
        "preprocessing procedure before running the evaluation; this notebook does not assume an imputation method."
    )

y_sample = df["label"].astype(int)
if not set(y_sample.unique()).issubset({0, 1}):
    raise ValueError(f"Expected binary labels 0/1, found: {sorted(y_sample.unique())}")

print("\nClass proportions:")
print(y_sample.value_counts(normalize=True))

## 4. Prepare the selected feature representation

- **Text:** `df["text"]` with TF-IDF.
- **Metadata:** the six metadata columns, passed through unchanged (`"passthrough"`). No `StandardScaler` is used in the main experiments.
- **Text + Metadata:** the `text` column and the six metadata columns (identifier and label columns are excluded), with TF-IDF applied to `text` and metadata passed through unchanged.

In [ ]:
if CONFIGURATION == "Text":
    X_sample = df["text"]
    preprocessor = TfidfVectorizer(
        max_features=TFIDF_MAX_FEATURES,
        ngram_range=TFIDF_NGRAM_RANGE,
    )

elif CONFIGURATION == "Metadata":
    X_sample = df[metadata_features]
    preprocessor = "passthrough"

else:  # CONFIGURATION == "Text + Metadata"
    X_sample = df[["text", *metadata_features]].copy()
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "text",
                TfidfVectorizer(
                    max_features=TFIDF_MAX_FEATURES,
                    ngram_range=TFIDF_NGRAM_RANGE,
                ),
                "text",
            ),
            ("meta", "passthrough", metadata_features),
        ]
    )

print("X type:", type(X_sample))
print("X shape:", X_sample.shape)
print("Labels:", y_sample.shape)
print("Preprocessor:", preprocessor)

## 5. Cross-validation setup

In [ ]:
cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
)

print(f"CV: {N_SPLITS} folds × {N_REPEATS} repetitions = {N_SPLITS * N_REPEATS} evaluation folds")

## 6. Classifiers and hyperparameters

The values below follow the final model dictionary in the uploaded notebook. They are not tuned or changed in this reorganization.

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(alpha=1.0),

    "Linear SVM": LinearSVC(
        C=1.0,
        max_iter=5000,
        random_state=42,
    ),

    "Logistic Regression": LogisticRegression(
        solver="saga",
        C=0.1,
        max_iter=20000,
        tol=1e-3,
        random_state=42,
        n_jobs=-1,
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        eval_metric="logloss",
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=1,
    ),
}

print("Models to evaluate:")
for model_name in models:
    print("-", model_name)

## 7. Run the selected configuration

Only the corpus and configuration selected in Section 2 are used. Each model is saved to the results list as it completes, and both the summary metrics and all-fold metrics are exported using dataset- and configuration-specific filenames (for example `DatasetA_Text_AllFoldMetrics.csv`).

In [ ]:
scoring = ["accuracy", "precision", "recall", "f1"]
results = []
fold_results = {}

configuration_tag = CONFIGURATION.replace(" + ", "_Plus_").replace(" ", "_")
summary_path = f"Dataset{DATASET}_{configuration_tag}_SummaryMetrics.csv"
folds_path = f"Dataset{DATASET}_{configuration_tag}_AllFoldMetrics.csv"

for name, model in models.items():
    print(f"\nRunning {name} on Dataset {DATASET} for configuration: {CONFIGURATION}...")

    pipeline = Pipeline([
        ("features", preprocessor),
        ("classifier", model),
    ])

    scores = cross_validate(
        pipeline,
        X_sample,
        y_sample,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
    )

    fold_results[f"{name}_Accuracy"] = scores["test_accuracy"]
    fold_results[f"{name}_Precision"] = scores["test_precision"]
    fold_results[f"{name}_Recall"] = scores["test_recall"]
    fold_results[f"{name}_F1"] = scores["test_f1"]

    result = {
        "Dataset": DATASET,
        "Configuration": CONFIGURATION,
        "Model": name,
        "Accuracy Mean": np.mean(scores["test_accuracy"]),
        "Accuracy Std": np.std(scores["test_accuracy"]),
        "Precision Mean": np.mean(scores["test_precision"]),
        "Precision Std": np.std(scores["test_precision"]),
        "Recall Mean": np.mean(scores["test_recall"]),
        "Recall Std": np.std(scores["test_recall"]),
        "F1 Mean": np.mean(scores["test_f1"]),
        "F1 Std": np.std(scores["test_f1"]),
    }

    results.append(result)
    pd.DataFrame(results).to_csv(summary_path, index=False)
    pd.DataFrame(fold_results).to_csv(folds_path, index=False)

    print(result)
    print(f"Updated: {summary_path}")
    print(f"Updated: {folds_path}")

## 8. Review the results

In [ ]:
results_df = pd.DataFrame(results)
results_df

## 9. Optional exploratory analysis — Logistic Regression with standardized metadata (Dataset A, Text + Metadata)

This section is **not part of the reported main experiments**. In a separate exploratory analysis limited to Logistic Regression on Dataset A with the combined Text + Metadata features, `StandardScaler` was applied to the six metadata columns with `with_mean=False` and `with_std=True`. The code is provided below as comments so that it can be executed if you want to repeat that analysis.

To run it: set `DATASET = "A"` and `CONFIGURATION = "Text + Metadata"` in Section 2, run Sections 1–6, then remove the leading `# ` from the lines of the next cell and run it.

In [ ]:
# ---------------------------------------------------------------------------------------------
# OPTIONAL EXPLORATORY ANALYSIS (not part of the reported main experiments)
# Logistic Regression, Dataset A, Text + Metadata, metadata standardized with
# StandardScaler(with_mean=False, with_std=True). Remove the leading "# " from each line to run.
# The scaler is part of the pipeline, so it is fitted on the training part of each fold only.
# ---------------------------------------------------------------------------------------------
# from sklearn.base import clone
# from sklearn.preprocessing import StandardScaler
#
# assert DATASET == "A" and CONFIGURATION == "Text + Metadata", (
#     'Set DATASET = "A" and CONFIGURATION = "Text + Metadata" in Section 2 and rerun Sections 1-6.'
# )
#
# preprocessor_scaled = ColumnTransformer(
#     transformers=[
#         (
#             "text",
#             TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, ngram_range=TFIDF_NGRAM_RANGE),
#             "text",
#         ),
#         ("meta", StandardScaler(with_mean=False, with_std=True), metadata_features),
#     ]
# )
#
# pipeline_scaled = Pipeline([
#     ("features", preprocessor_scaled),
#     ("classifier", clone(models["Logistic Regression"])),   # same hyperparameters as the main experiments
# ])
#
# scores_scaled = cross_validate(
#     pipeline_scaled,
#     X_sample,
#     y_sample,
#     cv=cv,
#     scoring=scoring,
#     n_jobs=1,
# )
#
# scaled_fold_scores = pd.DataFrame({
#     "LR_scaled_Accuracy": scores_scaled["test_accuracy"],
#     "LR_scaled_Precision": scores_scaled["test_precision"],
#     "LR_scaled_Recall": scores_scaled["test_recall"],
#     "LR_scaled_F1": scores_scaled["test_f1"],
# })
# scaled_fold_scores.to_csv("DatasetA_Text_Plus_Metadata_LR_StandardScaler_AllFoldMetrics.csv", index=False)
#
# print("Logistic Regression with standardized metadata (Dataset A, Text + Metadata)")
# print(scaled_fold_scores.agg(["mean", "std"]).round(4))

## Notes

1. To run a different setup, change `DATASET` (`"A"` or `"B"`) and/or `CONFIGURATION` (`"Text"`, `"Metadata"`, or `"Text + Metadata"`), then **restart the kernel and run all cells** so previous variables and outputs cannot interfere.
2. The main experiments use `"passthrough"` for metadata and do not use `StandardScaler`. The only analysis that standardizes the metadata is the optional exploratory Logistic Regression analysis in the final section, which is not part of the reported main experiments.
3. The input files are `dataset_a_within_30000_ids.pkl` and `dataset_b_within_30000_ids.pkl`; both must be present in `DATA_DIR`.
4. Before using the exports in the manuscript, confirm that the output matches the reported within-corpus tables; scikit-learn and XGBoost versions can change results slightly, so record the package versions you use.